# Second order uncertainty

This section of the notebook explores the implications of "second order uncertainty" when pooling evidence from multiple independent estimates of a parameter.  "Second order uncertainty" refers to the uncertainty in the estimated standard errors of the estimates being pooled.  It is distinct from the "first order uncertainty", which is captured through the standard errors themselves.  The second order uncertainty is expressed through degrees of freedom, where lower degrees of freedom corresponds to greater second order uncertainty.

In [ ]:
import numpy as np
import scipy.stats.distributions as dist

The cell below contains a function that estimates the standard error of the pooled estimate, where the pooling is done optimally by using inverse variance weights.  The function returns the naive and adjusted standard error and the degrees of freedom for the pooled estimate.

In [ ]:
def pooled_se(s, k):
    """
    Returns information needed for uncertainty analysis of the inverse variance
    weighted mean.
    
    Arguments:
    ----------
    s : The standard error of each estimate
    k : The degrees of freedom of each estimate
    
    Returns:
    --------
    se : The approximate standard error of the pooled estimate, accounting 
         for uncertainty in the weights
    se0 : The approximate standard error of the pooled estimate, ignoring 
          uncertainty in the weights
    dof : The degrees of freedom for the pooled estimate
    """
    assert(len(s) == len(k))
    assert(s.min() > 0)
    assert(k.min() > 0)
    m = len(s)
    v = s**2
    H = 1 / np.mean(1 / v)
    w = 1 / v
    w /= w.sum()
    kp = k - 4*(m-2)/(m-1)
    f = 1 + 4 * np.sum(w * (1 - w) / kp)
    se = np.sqrt(H*f/m)
    se0 = np.sqrt(H/m)
    dof = 1 / np.sum(w**2 / k)
    return se, se0, dof, np.sqrt(H)

Next we consider a series of examples, with different patterns of first-order and second-order uncertainty.

With high degrees of freedom, the naive and corrected standard errors are very similar, even when there is heterogeneity in the first order uncertainty.  Moreover, the pooled estimate has high degrees of freedom (though much smaller than the pooled sample size):

In [ ]:
s = np.r_[1, 2, 3, 4, 5, 6, 7]
k = np.r_[2000, 2000, 2000, 2000, 2000, 2000, 2000]
se, se0, dof, H = pooled_se(s, k)
q = dist.t(dof).ppf(0.975) / dist.norm().ppf(0.975)
print(se, se0, dof, se/se0, H, q)

With low degrees of freedom, the adjusted standard error is about 9% bigger than the 9 degrees of freedom.  Here we have no heterogeneity in the first order uncertainty, and the degrees of freedom is almost identical to the pooled sample size.

In [ ]:
s = np.r_[1, 1, 1, 1, 1]
k = np.r_[20, 20, 20, 20, 20]
se, se0, dof, H = pooled_se(s, k)
q = dist.t(dof).ppf(0.975) / dist.norm().ppf(0.975)
print(se, se0, dof, se/se0, H, q)

The next example is similar to the previous one, except with more studies.  This increases the degrees of freedom of the pooled estimate, but the adjusted and naive standard errors remain similar. 

In [ ]:
s = np.r_[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
k = np.r_[20, 20, 20, 20, 20, 20, 20, 20, 20, 20]
se, se0, dof, H = pooled_se(s, k)
q = dist.t(dof).ppf(0.975) / dist.norm().ppf(0.975)
print(se, se0, dof, se/se0, H, q)

Next we consider a situation with heterogeneity in the first order uncertainty, and moderate but homogeneous second order uncertainty.  The adjusted SE is about 6% greater than the naive SE, and the pooled degrees of freedom is sufficiently low that our confidence intervals will be about 3% wider for this reason alone.

In [ ]:
s = np.r_[1, 2, 3, 4, 5]
k = np.r_[20, 20, 20, 20, 20]
se, se0, dof, H = pooled_se(s, k)
q = dist.t(dof).ppf(0.975) / dist.norm().ppf(0.975)
print(se, se0, dof, se/se0, H, q)

In [ ]:
s = np.r_[1, 1, 1, 1, 1]
k = np.r_[20, 30, 40, 50, 60]
se, se0, dof, H = pooled_se(s, k)
q = dist.t(dof).ppf(0.975) / dist.norm().ppf(0.975)
print(se, se0, dof, se/se0, H, q)

Next we consider a situation where there is heterogeneity in both the first and second order uncertainties, and this heterogeneity follows a pattern that is correlated between the two (which is likely to be the case in practice).  The resulats are similar to what we saw above when there was no heterogeneity in the second order uncertainty.

In [ ]:
s = np.r_[1, 2, 3, 4, 5]
k = np.r_[20, 30, 40, 50, 60]
se, se0, dof, H = pooled_se(s, k)
q = dist.t(dof).ppf(0.975) / dist.norm().ppf(0.975)
print(se, se0, dof, se/se0, H, q)

Next we consider a situation where there is heterogeneity in both the first and second order uncertainty, but the patterns of heterogeneity are anticorrelated.  This is unlikely in practice, and leads to more benign inflation of the pooled uncertainty compared to the situation where the two orders of uncertainty are correlated.

In [ ]:
s = np.r_[5, 4, 3, 2, 1]
k = np.r_[20, 30, 40, 50, 60]
se, se0, dof, H = pooled_se(s, k)
q = dist.t(dof).ppf(0.975) / dist.norm().ppf(0.975)
print(se, se0, dof, se/se0, H, q)

# p-value combining rules

This section of the notebook explores methods for combining p-values, focusing on the classical "Fisher" approach, and the much more recent "Cauchy" approach, with the latter producing valid results even if the p-values are not independent.  

First, we implement the two methods.

In [ ]:
def fisher_combine(p):
    """
    Produce a meta p-value for the p-values in 'p', using the Fisher combining rule.
    """
    m = len(p)
    f = -2*np.sum(np.log(p))
    mp = 1 - dist.chi2(2*m).cdf(f)
    return mp

In [ ]:
def cauchy_combine(p):
    """
    Produce a meta p-value for the p-values in 'p', using the Cauchy combining rule.
    """
    T = np.sum(np.tan(np.pi*(1/2-p)))
    return 1 - dist.cauchy().cdf(T)

Next we show that in the case where there is only a single p-value, both methods give back this same p-value as the combined result.

In [ ]:
p = np.r_[0.01]
fc = fisher_combine(p)
cc = cauchy_combine(p)
fc, cc

Next we consider a setting where we have several studies that give identical levels of evidence against the null hypothesis.

In [ ]:
p = np.r_[0.05, 0.05, 0.05, 0.05]
fc = fisher_combine(p)
cc = cauchy_combine(p)
fc, cc

Next we consider a situation where one study has much less evidence against the null than the others.

In [ ]:
p = np.r_[0.05, 0.05, 0.05, 0.9]
fc = fisher_combine(p)
cc = cauchy_combine(p)
fc, cc

Next we consider a setting where one study has much more evidence against the null than the others.

In [ ]:
p = np.r_[0.005, 0.5, 0.5, 0.9]
fc = fisher_combine(p)
cc = cauchy_combine(p)
fc, cc

In [ ]:
p = np.r_[0.01, 0.05, 0.1, 0.5, 0.9]
fc = fisher_combine(p)
cc = cauchy_combine(p)
fc, cc